# Guarded Section-wise LongT5 v6 — Colab Test Notebook Without Gradio

This version does **not** launch a UI.  
You run the summarizer directly inside notebook cells and see printed output.

Recommended runtime:  
`Runtime → Change runtime type → T4 GPU`


In [ ]:
# Cell 1 — Install dependencies
!pip -q install -U \
  transformers \
  sentence-transformers \
  scikit-learn \
  pymupdf \
  accelerate \
  safetensors \
  protobuf \
  sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 22.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.34.1 which is incompatible.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires prot

In [ ]:
# Cell 2 — Check runtime
import torch, platform

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Use Runtime → Change runtime type → T4 GPU.")


Python: 3.12.13
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# Cell 3 — Model configuration
# You can replace SUM_MODEL_NAME later with your fine-tuned Legal LongT5 model.

import os

os.environ["EMBED_MODEL_NAME"] = "sentence-transformers/all-MiniLM-L6-v2"
os.environ["SUM_MODEL_NAME"] = "pszemraj/long-t5-tglobal-base-16384-book-summary"

print("Embedding model:", os.environ["EMBED_MODEL_NAME"])
print("Summarization model:", os.environ["SUM_MODEL_NAME"])


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Summarization model: pszemraj/long-t5-tglobal-base-16384-book-summary


## Load the summarization pipeline

This cell defines all functions from your app, but removes Gradio completely.


In [ ]:
import os
import re
from collections import Counter
from typing import List, Tuple

import numpy as np
try:
    import pymupdf
except ImportError:
    import fitz as pymupdf  # fallback for older PyMuPDF builds
import torch
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# =========================================================
# Guarded Section-wise LongT5 v6
# =========================================================
#
# Main upgrades:
# 1. Conservative section detection.
# 2. Input and output quality guards.
# 3. Extractive factual/legal skeleton.
# 4. LongT5 section/window summaries.
# 5. Final LongT5 re-summary.
# 6. Bad-output retry.
# 7. Extractive fallback if LongT5 output is still bad.
# 8. Optional K-Means/MMR anchors only as guidance.
#
# K-Means never reorders the document and never summarizes clusters.
# =========================================================


# -----------------------------
# Config
# -----------------------------
EMBED_MODEL_NAME = os.getenv("EMBED_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")

# LongT5-only generation model.
# Replace this later with your fine-tuned legal LongT5 model:
# SUM_MODEL_NAME=Harshit0502/legal-longt5-sectionwise
SUM_MODEL_NAME = os.getenv("SUM_MODEL_NAME", "pszemraj/long-t5-tglobal-base-16384-book-summary")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

_embedder = None
_tokenizer = None
_model = None


# -----------------------------
# Model loading
# -----------------------------
def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
    return _embedder


def get_longt5():
    global _tokenizer, _model

    if _tokenizer is None or _model is None:
        _tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL_NAME)
        dtype = torch.float16 if DEVICE == "cuda" else torch.float32

        # Stable loader: do not use low_cpu_mem_usage=True + .to(DEVICE)
        _model = AutoModelForSeq2SeqLM.from_pretrained(
            SUM_MODEL_NAME,
            torch_dtype=dtype,
        )
        _model = _model.to(DEVICE)
        _model.eval()

    return _tokenizer, _model


# -----------------------------
# PDF extraction and cleaning
# -----------------------------
def normalize_line(line: str) -> str:
    line = re.sub(r"\d+", "#", line)
    line = re.sub(r"\s+", " ", line)
    return line.strip().lower()


def clean_pdf_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\r", "\n").replace("\t", " ")
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\bwww\.\S+", " ", text)
    text = re.sub(r"\n?\s*page\s+\d+\s*(of\s+\d+)?\s*\n?", "\n", text, flags=re.I)
    text = re.sub(r"\n?\s*\d+\s*/\s*\d+\s*\n?", "\n", text)

    # Remove common digital signature noise but do not over-clean legal content.
    text = re.sub(r"Digitally signed by.*?Reason:\s*Signature Not Verified", " ", text, flags=re.I | re.S)
    text = re.sub(r"Signature Not Verified", " ", text, flags=re.I)

    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def remove_run_repetition(text: str, max_run: int = 3) -> str:
    """
    Removes absurd repeated word runs such as:
    us us us us, kids kids kids, with with with...
    Keeps at most max_run consecutive occurrences.
    """
    words = text.split()
    if not words:
        return text

    out = []
    last = None
    run = 0

    for w in words:
        key = re.sub(r"\W+", "", w.lower())
        if key and key == last:
            run += 1
        else:
            last = key
            run = 1

        if run <= max_run:
            out.append(w)

    return " ".join(out)


def extract_text_from_pdf(pdf_file, max_pages: int = 30) -> Tuple[str, str]:
    if pdf_file is None:
        return "", "No PDF uploaded."

    pdf_path = pdf_file.name if hasattr(pdf_file, "name") else str(pdf_file)

    try:
        doc = pymupdf.open(pdf_path)
        total_pages = min(len(doc), int(max_pages))

        page_lines = []
        for idx in range(total_pages):
            page_text = doc[idx].get_text("text") or ""
            lines = [line.strip() for line in page_text.splitlines() if line.strip()]
            page_lines.append(lines)

        doc.close()

        # Repeated header/footer removal.
        line_counter = Counter()
        for lines in page_lines:
            unique = set()
            for line in lines:
                norm = normalize_line(line)
                if 3 <= len(norm) <= 150:
                    unique.add(norm)
            line_counter.update(unique)

        threshold = max(2, int(0.30 * total_pages))
        repeated = {line for line, count in line_counter.items() if count >= threshold}

        cleaned_pages = []
        for page_num, lines in enumerate(page_lines, start=1):
            kept = []
            for line in lines:
                norm = normalize_line(line)
                if norm in repeated:
                    continue
                if re.fullmatch(r"\d+", line):
                    continue
                if len(line) <= 2:
                    continue
                kept.append(line)

            if kept:
                cleaned_pages.append(f"[PAGE {page_num}]\n" + "\n".join(kept))

        extracted = clean_pdf_text("\n\n".join(cleaned_pages))
        extracted = remove_run_repetition(extracted, max_run=3)

        q = text_quality_report(extracted)

        debug = (
            f"Pages read: {total_pages}\n"
            f"Extracted words: {len(extracted.split())}\n"
            f"Repeated header/footer lines removed: {len(repeated)}\n"
            f"Input quality: {q}"
        )

        if not extracted:
            return "No selectable text found. This PDF may be scanned and may need OCR.", debug

        return extracted, debug

    except Exception as e:
        return "", f"PDF extraction failed: {type(e).__name__}: {e}"


# -----------------------------
# Text utilities
# -----------------------------
def clean_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\r", "\n").replace("\t", " ")
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_into_sentences(text: str) -> List[str]:
    text = clean_text(text)
    text = re.sub(r"\s+", " ", text)

    pieces = re.split(r"(?<=[.!?])\s+(?=[A-Z\[\(0-9])", text)
    sentences = []

    for p in pieces:
        p = p.strip()
        # Filter tiny fragments and obvious page markers.
        if len(p.split()) >= 7 and not re.fullmatch(r"\[PAGE \d+\]", p.strip(), flags=re.I):
            sentences.append(p)

    # Fallback: paragraph-level units
    if len(sentences) < 4:
        paras = [p.strip() for p in text.split("\n") if len(p.strip().split()) >= 10]
        return paras

    return sentences


def token_len(text: str) -> int:
    tokenizer, _ = get_longt5()
    return len(tokenizer.encode(text, add_special_tokens=False))


def split_token_windows(text: str, max_tokens: int = 3072, overlap_tokens: int = 96) -> List[str]:
    tokenizer, _ = get_longt5()
    ids = tokenizer.encode(clean_text(text), add_special_tokens=False)

    if not ids:
        return []

    max_tokens = int(max_tokens)
    overlap_tokens = int(overlap_tokens)
    step = max(1, max_tokens - overlap_tokens)

    windows = []
    for start in range(0, len(ids), step):
        sub_ids = ids[start:start + max_tokens]
        window = tokenizer.decode(sub_ids, skip_special_tokens=True)
        window = clean_text(window)
        if window:
            windows.append(window)

    return windows


def minmax(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    denom = float(x.max() - x.min())
    if denom < 1e-9:
        return np.ones_like(x, dtype=np.float32) * 0.5
    return (x - x.min()) / denom


# -----------------------------
# Quality guards
# -----------------------------
LEGAL_KEYWORDS = {
    "court", "appeal", "appellant", "respondent", "petitioner", "accused",
    "witness", "evidence", "fir", "police", "investigation", "trial",
    "conviction", "sentence", "acquitted", "section", "ipc", "crpc",
    "judgment", "order", "offence", "prosecution", "deceased", "injury",
    "high", "supreme", "state", "law", "criminal", "civil", "learned"
}


def text_quality_report(text: str) -> str:
    words = re.findall(r"[A-Za-z0-9@./'-]+", clean_text(text).lower())
    if not words:
        return "bad: empty"

    unique_ratio = len(set(words)) / max(1, len(words))
    top_word, top_count = Counter(words).most_common(1)[0]
    top_word_ratio = top_count / max(1, len(words))
    alpha_ratio = sum(1 for w in words if re.search(r"[a-z]", w)) / max(1, len(words))
    longish_ratio = sum(1 for w in words if len(w) >= 4) / max(1, len(words))
    legal_hits = sum(1 for w in words if w in LEGAL_KEYWORDS)

    bad_flags = []
    if len(words) < 50:
        bad_flags.append("too_short")
    if unique_ratio < 0.25:
        bad_flags.append("low_unique")
    if top_word_ratio > 0.14:
        bad_flags.append("top_word_repeated")
    if alpha_ratio < 0.55:
        bad_flags.append("low_alpha")
    if longish_ratio < 0.35:
        bad_flags.append("too_many_short_tokens")

    status = "bad" if bad_flags else "ok"

    return (
        f"{status}; words={len(words)}, unique_ratio={unique_ratio:.2f}, "
        f"top_word='{top_word}'({top_word_ratio:.2f}), legal_hits={legal_hits}, "
        f"flags={bad_flags if bad_flags else 'none'}"
    )


def is_bad_text(text: str) -> bool:
    report = text_quality_report(text)
    return report.startswith("bad")


def is_bad_summary(text: str) -> Tuple[bool, str]:
    text = clean_text(text)
    words = re.findall(r"[A-Za-z0-9@./'-]+", text.lower())

    if len(words) < 35:
        return True, "too_short"

    unique_ratio = len(set(words)) / max(1, len(words))
    counter = Counter(words)
    top_word, top_count = counter.most_common(1)[0]
    top_word_ratio = top_count / max(1, len(words))

    # Repeated adjacent words or repeated 2-grams are strong garbage signs.
    repeated_adjacent = sum(1 for i in range(len(words) - 1) if words[i] == words[i + 1])

    bigrams = list(zip(words, words[1:]))
    bigram_counts = Counter(bigrams)
    repeated_bigram_ratio = 0.0
    if bigrams:
        repeated_bigram_ratio = bigram_counts.most_common(1)[0][1] / len(bigrams)

    legal_hits = sum(1 for w in words if w in LEGAL_KEYWORDS)

    if unique_ratio < 0.32:
        return True, f"low_unique_ratio={unique_ratio:.2f}"
    if top_word_ratio > 0.12:
        return True, f"top_word_repeated={top_word}:{top_word_ratio:.2f}"
    if repeated_adjacent > 6:
        return True, f"repeated_adjacent={repeated_adjacent}"
    if repeated_bigram_ratio > 0.08:
        return True, f"repeated_bigram_ratio={repeated_bigram_ratio:.2f}"
    if legal_hits == 0 and len(words) > 120:
        return True, "no_legal_terms"

    return False, "ok"


def final_polish(summary_text: str) -> str:
    text = clean_text(summary_text)
    text = re.sub(r"\b(www|http)\S+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)

    # Remove exact duplicated sentences.
    sentences = split_into_sentences(text)
    seen = set()
    unique = []

    for sentence in sentences:
        key = re.sub(r"\W+", "", sentence.lower())[:130]
        if key and key not in seen:
            seen.add(key)
            unique.append(sentence)

    if unique:
        text = " ".join(unique)

    return clean_text(text)


# -----------------------------
# Conservative section detection
# -----------------------------
SECTION_RULES = [
    ("Facts / Background", [
        r"facts?", r"brief facts", r"facts of the case", r"factual background",
        r"background", r"material facts", r"prosecution case"
    ]),
    ("Issues / Questions", [
        r"issues?", r"points? for determination", r"questions? for consideration",
        r"questions? of law", r"substantial questions? of law"
    ]),
    ("Petitioner/Appellant Arguments", [
        r"petitioner'?s? submissions?", r"appellant'?s? submissions?",
        r"arguments? on behalf of the petitioner", r"arguments? on behalf of the appellant",
        r"contentions? of the petitioner", r"contentions? of the appellant"
    ]),
    ("Respondent Arguments", [
        r"respondent'?s? submissions?", r"state'?s? submissions?",
        r"arguments? on behalf of the respondent", r"arguments? on behalf of the state",
        r"contentions? of the respondent", r"contentions? of the state"
    ]),
    ("Submissions / Arguments", [
        r"submissions?", r"arguments?", r"contentions?", r"rival submissions?"
    ]),
    ("Evidence / Record", [
        r"evidence", r"evidence on record", r"material on record", r"record"
    ]),
    ("Reasoning / Analysis", [
        r"analysis", r"discussion", r"consideration", r"findings?", r"reasons?",
        r"reasoning", r"discussion and findings", r"analysis and findings"
    ]),
    ("Decision / Order", [
        r"order", r"final order", r"operative order", r"conclusion", r"decision",
        r"result", r"relief", r"appeals are allowed", r"appeal is allowed",
        r"appeal dismissed", r"appeals are dismissed"
    ]),
]


def clean_heading_candidate(line: str) -> str:
    line = line.strip()
    line = re.sub(r"^\[PAGE\s+\d+\]\s*", "", line, flags=re.I)
    line = re.sub(r"^\d+[\.\)]\s*", "", line)
    line = re.sub(r"^[IVXLC]+[\.\)]\s*", "", line, flags=re.I)
    line = line.strip(":-–—. ")
    line = re.sub(r"\s+", " ", line)
    return line


def canonical_section_label(line: str) -> str:
    cleaned = clean_heading_candidate(line).lower()

    # Conservative: require short heading-like line.
    if len(cleaned.split()) > 10:
        return ""

    for label, patterns in SECTION_RULES:
        for p in patterns:
            if re.fullmatch(p, cleaned, flags=re.I):
                return label

    return ""


def detect_sections(text: str, min_section_words: int = 80) -> Tuple[List[Tuple[str, str]], str]:
    text = clean_text(text)
    lines = text.splitlines()

    heading_positions = []

    for i, line in enumerate(lines):
        label = canonical_section_label(line)
        if label:
            heading_positions.append((i, label, clean_heading_candidate(line)))

    # Remove near duplicates.
    filtered = []
    last_i = -999
    for item in heading_positions:
        i, label, raw = item
        if filtered and i - last_i <= 1 and filtered[-1][1] == label:
            continue
        filtered.append(item)
        last_i = i

    heading_positions = filtered

    debug = [f"Conservative heading matches: {len(heading_positions)}"]

    if len(heading_positions) < 2:
        debug.append("Section detection weak → fallback required.")
        return [("Full Document", text)], "\n".join(debug)

    sections = []
    first_line = heading_positions[0][0]

    if first_line > 0:
        intro = "\n".join(lines[:first_line]).strip()
        if len(intro.split()) >= min_section_words and not is_bad_text(intro):
            sections.append(("Introduction / Case Metadata", intro))

    for idx, (start, label, raw) in enumerate(heading_positions):
        end = heading_positions[idx + 1][0] if idx + 1 < len(heading_positions) else len(lines)
        content = "\n".join(lines[start:end]).strip()

        if len(content.split()) >= min_section_words and not is_bad_text(content):
            sections.append((label, content))
        else:
            debug.append(f"Rejected weak/noisy section: {label} ({len(content.split())} words)")

    if len(sections) < 2:
        debug.append("Valid sections < 2 → fallback required.")
        return [("Full Document", text)], "\n".join(debug)

    debug.append(f"Valid sections used: {len(sections)}")
    for idx, (label, content) in enumerate(sections, start=1):
        debug.append(f"{idx}. {label}: {len(content.split())} words")

    return sections, "\n".join(debug)


# -----------------------------
# Extractive factual skeleton
# -----------------------------
def extract_legal_skeleton(
    text: str,
    max_sentences: int = 8,
    use_minilm: bool = True,
) -> Tuple[str, str]:
    """
    Extracts important diverse sentences to ground LongT5.
    This is the main anti-hallucination helper.
    """
    sentences = split_into_sentences(text)

    if not sentences:
        return "", "No sentences for skeleton."

    # Cap for speed.
    if len(sentences) > 900:
        stride = max(1, len(sentences) // 900)
        sentences = sentences[::stride]

    try:
        # TF-IDF salience against local document.
        vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=12000,
            min_df=1,
        )
        local_doc = " ".join(sentences)
        tfidf = vectorizer.fit_transform(sentences + [local_doc])
        sent_tfidf = tfidf[:-1]
        doc_tfidf = tfidf[-1]
        tfidf_scores = cosine_similarity(sent_tfidf, doc_tfidf).reshape(-1)
        relevance = minmax(tfidf_scores)

        # Boost legal-keyword sentences.
        legal_boost = []
        for s in sentences:
            words = re.findall(r"[A-Za-z]+", s.lower())
            hits = sum(1 for w in words if w in LEGAL_KEYWORDS)
            legal_boost.append(min(1.0, hits / 3.0))
        legal_boost = np.asarray(legal_boost, dtype=np.float32)

        if use_minilm and len(sentences) >= 3:
            embedder = get_embedder()
            emb = embedder.encode(
                sentences,
                convert_to_numpy=True,
                normalize_embeddings=True,
                batch_size=32,
                show_progress_bar=False,
            ).astype(np.float32)

            centroid = emb.mean(axis=0, keepdims=True)
            semantic_scores = minmax(cosine_similarity(emb, centroid).reshape(-1))
            base_score = 0.55 * relevance + 0.25 * semantic_scores + 0.20 * legal_boost
        else:
            emb = None
            base_score = 0.75 * relevance + 0.25 * legal_boost

        # MMR-style diverse selection.
        selected = []
        candidates = list(range(len(sentences)))
        lambda_rel = 0.75

        while candidates and len(selected) < int(max_sentences):
            best_idx = None
            best_score = -1e9

            for idx in candidates:
                if emb is not None and selected:
                    redundancy = max(cosine_similarity(emb[idx:idx+1], emb[selected]).reshape(-1))
                else:
                    redundancy = 0.0

                score = lambda_rel * base_score[idx] - (1 - lambda_rel) * redundancy

                # Light preference to first/last 15% of a section/document.
                pos = idx / max(1, len(sentences) - 1)
                if pos < 0.15 or pos > 0.85:
                    score += 0.04

                if score > best_score:
                    best_score = score
                    best_idx = idx

            selected.append(best_idx)
            candidates.remove(best_idx)

        selected = sorted(selected)

        skeleton_lines = [
            f"{i+1}. {sentences[idx]}"
            for i, idx in enumerate(selected)
        ]

        return "\n".join(skeleton_lines), f"Skeleton sentences used: {len(skeleton_lines)}"

    except Exception as e:
        return "", f"Skeleton extraction failed safely: {type(e).__name__}: {e}"


# -----------------------------
# Optional K-Means / MMR anchors inside a section/window
# -----------------------------
def kmeans_anchors_for_text(
    text: str,
    anchors_count: int = 3,
    minilm_weight: float = 0.60,
    tfidf_weight: float = 0.40,
) -> Tuple[str, str]:
    sentences = split_into_sentences(text)

    if len(sentences) < 4:
        return "", "K-Means anchors skipped: too few sentences."

    max_sentences = 120
    if len(sentences) > max_sentences:
        stride = max(1, len(sentences) // max_sentences)
        sentences = sentences[::stride]

    k = max(1, min(int(anchors_count), len(sentences)))

    try:
        embedder = get_embedder()
        dense = embedder.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=32,
            show_progress_bar=False,
        ).astype(np.float32)

        vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=8000,
            min_df=1,
        )
        tfidf = vectorizer.fit_transform(sentences)
        max_dim = min(tfidf.shape[0] - 1, tfidf.shape[1] - 1, 64)

        if max_dim >= 2:
            svd = TruncatedSVD(n_components=max_dim, random_state=42)
            sparse = svd.fit_transform(tfidf).astype(np.float32)
        else:
            sparse = tfidf.toarray().astype(np.float32)

        sparse = normalize(sparse)

        total = max(1e-6, float(minilm_weight) + float(tfidf_weight))
        minilm_weight = float(minilm_weight) / total
        tfidf_weight = float(tfidf_weight) / total

        hybrid = normalize(np.concatenate([minilm_weight * dense, tfidf_weight * sparse], axis=1))

        if k == 1:
            labels = np.zeros(len(sentences), dtype=int)
            centers = np.mean(hybrid, axis=0, keepdims=True)
        else:
            km = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = km.fit_predict(hybrid)
            centers = km.cluster_centers_

        chosen = []
        for cluster_id in sorted(set(labels)):
            idxs = np.where(labels == cluster_id)[0]
            if len(idxs) == 0:
                continue
            center = centers[cluster_id].reshape(1, -1)
            sim = cosine_similarity(hybrid[idxs], center).reshape(-1)
            chosen.append(int(idxs[int(np.argmax(sim))]))

        chosen = sorted(set(chosen))
        anchors = [f"{i+1}. {sentences[idx]}" for i, idx in enumerate(chosen)]

        return "\n".join(anchors), f"K-Means anchors used: {len(anchors)}"

    except Exception as e:
        return "", f"K-Means anchors failed safely: {type(e).__name__}: {e}"


def mmr_anchors_for_text(text: str, anchors_count: int = 3) -> Tuple[str, str]:
    skeleton, debug = extract_legal_skeleton(text, max_sentences=anchors_count, use_minilm=True)
    return skeleton, debug.replace("Skeleton", "MMR anchor")


def get_anchors(text: str, method: str, anchors_count: int, minilm_weight: float, tfidf_weight: float) -> Tuple[str, str]:
    if method == "K-Means":
        return kmeans_anchors_for_text(text, anchors_count, minilm_weight, tfidf_weight)
    if method == "MMR":
        return mmr_anchors_for_text(text, anchors_count)
    return "", "Anchors disabled."


# -----------------------------
# LongT5 generation with guard
# -----------------------------
@torch.inference_mode()
def longt5_generate_raw(
    text: str,
    max_input_tokens: int = 3072,
    max_output_tokens: int = 256,
    num_beams: int = 2,
    min_new_tokens: int = 40,
) -> str:
    tokenizer, model = get_longt5()
    text = clean_text(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=int(max_input_tokens),
    ).to(DEVICE)

    out = model.generate(
        **inputs,
        max_new_tokens=int(max_output_tokens),
        min_new_tokens=int(min_new_tokens),
        num_beams=int(num_beams),
        do_sample=False,
        no_repeat_ngram_size=3,
        repetition_penalty=1.12,
        length_penalty=1.05,
        early_stopping=True,
    )

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


def longt5_generate_guarded(
    prompt: str,
    fallback_text: str,
    max_input_tokens: int,
    max_output_tokens: int,
    num_beams: int,
    min_new_tokens: int,
    enable_retry: bool = True,
) -> Tuple[str, str]:
    """
    Generate with output quality guard.
    If rubbish, retry greedily with simpler source text.
    If still bad, return extractive fallback.
    """
    debug_lines = []

    summary = longt5_generate_raw(
        prompt,
        max_input_tokens=max_input_tokens,
        max_output_tokens=max_output_tokens,
        num_beams=num_beams,
        min_new_tokens=min_new_tokens,
    )
    summary = final_polish(summary)

    bad, reason = is_bad_summary(summary)
    debug_lines.append(f"first_generation_quality: {'bad' if bad else 'ok'} ({reason})")

    if not bad:
        return summary, "\n".join(debug_lines)

    if enable_retry:
        skeleton, sdebug = extract_legal_skeleton(
            fallback_text,
            max_sentences=max(4, min(10, max_output_tokens // 45)),
            use_minilm=False,
        )

        retry_prompt = (
            "Summarize this legal/document text faithfully using the factual skeleton. "
            "Do not invent facts. Avoid repetition.\n\n"
            f"FACTUAL SKELETON:\n{skeleton}\n\n"
            f"TEXT:\n{fallback_text}"
        )

        retry = longt5_generate_raw(
            retry_prompt,
            max_input_tokens=max_input_tokens,
            max_output_tokens=max_output_tokens,
            num_beams=1,
            min_new_tokens=min_new_tokens,
        )
        retry = final_polish(retry)

        bad2, reason2 = is_bad_summary(retry)
        debug_lines.append(f"retry_generation_quality: {'bad' if bad2 else 'ok'} ({reason2})")

        if not bad2:
            return retry, "\n".join(debug_lines)

    # Extractive fallback
    fallback, fb_debug = extract_legal_skeleton(
        fallback_text,
        max_sentences=max(5, min(12, max_output_tokens // 40)),
        use_minilm=False,
    )

    if not fallback:
        fallback = fallback_text[:1500]

    debug_lines.append(f"extractive_fallback_used: {fb_debug}")
    return fallback, "\n".join(debug_lines)


def group_token_safe(blocks: List[str], max_input_tokens: int, reserve_tokens: int = 192) -> List[str]:
    tokenizer, _ = get_longt5()

    groups = []
    current = []
    current_tokens = 0
    safe_limit = max(512, int(max_input_tokens) - int(reserve_tokens))

    for idx, block in enumerate(blocks, start=1):
        labelled = f"[ORDERED BLOCK {idx}]\n{block}"
        t = len(tokenizer.encode(labelled, add_special_tokens=False))

        if current and current_tokens + t > safe_limit:
            groups.append("\n\n".join(current))
            current = [labelled]
            current_tokens = t
        else:
            current.append(labelled)
            current_tokens += t

    if current:
        groups.append("\n\n".join(current))

    return groups


# -----------------------------
# Section/window summarization
# -----------------------------
def build_section_prompt(
    section_title: str,
    section_text: str,
    skeleton: str,
    anchors: str,
    anchor_method: str,
    part_info: str = "",
) -> str:
    title_line = f"Section: {section_title}"
    if part_info:
        title_line += f" ({part_info})"

    parts = [
        f"Summarize this legal/document section faithfully. {title_line}.",
        "The full section text is the source of truth. Do not invent facts. Avoid repetition.",
    ]

    if skeleton:
        parts.append(f"\nFACTUAL SKELETON:\n{skeleton}")

    if anchors and anchor_method != "None":
        parts.append(f"\nCOVERAGE ANCHORS ({anchor_method}, guidance only):\n{anchors}")

    parts.append(f"\nFULL SECTION TEXT:\n{section_text}")

    return "\n".join(parts)


def summarize_one_block(
    title: str,
    text: str,
    anchor_method: str,
    anchors_per_block: int,
    max_window_tokens: int,
    summary_tokens: int,
    beams: int,
    minilm_weight: float,
    tfidf_weight: float,
    use_skeleton: bool,
    part_info: str = "",
) -> Tuple[str, str]:
    debug = []

    skeleton = ""
    if use_skeleton:
        skeleton, sdebug = extract_legal_skeleton(text, max_sentences=6, use_minilm=False)
        debug.append(sdebug)

    anchors, adebug = get_anchors(text, anchor_method, anchors_per_block, minilm_weight, tfidf_weight)
    debug.append(adebug)

    prompt = build_section_prompt(
        section_title=title,
        section_text=text,
        skeleton=skeleton,
        anchors=anchors,
        anchor_method=anchor_method,
        part_info=part_info,
    )

    summary, gdebug = longt5_generate_guarded(
        prompt=prompt,
        fallback_text=text,
        max_input_tokens=int(max_window_tokens),
        max_output_tokens=int(summary_tokens),
        num_beams=int(beams),
        min_new_tokens=max(40, int(summary_tokens) // 5),
        enable_retry=True,
    )

    debug.append(gdebug)

    return summary, " | ".join(debug)


def summarize_section(
    title: str,
    content: str,
    anchor_method: str,
    anchors_per_block: int,
    max_window_tokens: int,
    overlap_tokens: int,
    section_summary_tokens: int,
    reduce_input_tokens: int,
    beams: int,
    minilm_weight: float,
    tfidf_weight: float,
    use_skeleton: bool,
) -> Tuple[str, str]:
    content = clean_text(content)
    debug = [f"'{title}': words={len(content.split())}, tokens={token_len(content)}"]

    if is_bad_text(content):
        debug.append(f"Rejected noisy content: {text_quality_report(content)}")
        fallback, fbdebug = extract_legal_skeleton(content, max_sentences=6, use_minilm=False)
        return fallback or "", "\n".join(debug + [fbdebug])

    if token_len(content) <= int(max_window_tokens):
        summary, sdebug = summarize_one_block(
            title=title,
            text=content,
            anchor_method=anchor_method,
            anchors_per_block=int(anchors_per_block),
            max_window_tokens=int(max_window_tokens),
            summary_tokens=int(section_summary_tokens),
            beams=int(beams),
            minilm_weight=float(minilm_weight),
            tfidf_weight=float(tfidf_weight),
            use_skeleton=use_skeleton,
        )
        debug.append(sdebug)
        return summary, "\n".join(debug)

    # Long section → ordered windows.
    windows = split_token_windows(content, max_tokens=max_window_tokens, overlap_tokens=overlap_tokens)
    debug.append(f"Long section split into {len(windows)} window(s).")

    window_summaries = []

    for idx, window in enumerate(windows, start=1):
        summary, wdebug = summarize_one_block(
            title=title,
            text=window,
            anchor_method=anchor_method,
            anchors_per_block=int(anchors_per_block),
            max_window_tokens=int(max_window_tokens),
            summary_tokens=int(section_summary_tokens),
            beams=int(beams),
            minilm_weight=float(minilm_weight),
            tfidf_weight=float(tfidf_weight),
            use_skeleton=use_skeleton,
            part_info=f"part {idx} of {len(windows)}",
        )
        window_summaries.append(summary)
        debug.append(f"window {idx}: {wdebug}")

    # Reduce windows to one section summary.
    current = [f"[{title} WINDOW {i+1}]\n{s}" for i, s in enumerate(window_summaries)]
    rounds = 0

    while True:
        merged = "\n\n".join(current)
        if token_len(merged) <= int(reduce_input_tokens) or len(current) <= 1:
            break

        groups = group_token_safe(current, max_input_tokens=reduce_input_tokens)
        reduced = []

        for group in groups:
            prompt = (
                f"Merge these ordered summaries from section '{title}' into one faithful section summary. "
                "Remove repetition and do not invent facts.\n\n"
                + group
            )

            red, rdebug = longt5_generate_guarded(
                prompt=prompt,
                fallback_text=group,
                max_input_tokens=int(reduce_input_tokens),
                max_output_tokens=max(180, int(section_summary_tokens)),
                num_beams=int(beams),
                min_new_tokens=60,
                enable_retry=True,
            )
            reduced.append(red)
            debug.append(f"section reduce: {rdebug}")

        current = [f"[REDUCED {i+1}]\n{s}" for i, s in enumerate(reduced)]
        rounds += 1
        if rounds >= 4:
            break

    final_section_input = "\n\n".join(current)

    final_prompt = (
        f"Create one coherent summary for section '{title}' from these ordered summaries. "
        "Do not invent facts. Avoid repetition.\n\n"
        + final_section_input
    )

    final_summary, fdebug = longt5_generate_guarded(
        prompt=final_prompt,
        fallback_text=final_section_input,
        max_input_tokens=int(reduce_input_tokens),
        max_output_tokens=int(section_summary_tokens),
        num_beams=int(beams),
        min_new_tokens=max(60, int(section_summary_tokens) // 4),
        enable_retry=True,
    )

    debug.append(f"section final reduce rounds={rounds}; {fdebug}")

    return final_summary, "\n".join(debug)


# -----------------------------
# Main pipeline
# -----------------------------
def guarded_sectionwise_summarize(
    input_text: str,
    mode: str,
    force_ordered_windows: bool,
    anchor_method: str,
    anchors_per_block: int,
    use_skeleton: bool,
    max_window_tokens: int,
    overlap_tokens: int,
    section_summary_tokens: int,
    reduce_input_tokens: int,
    final_summary_tokens: int,
    minilm_weight: float,
    tfidf_weight: float,
) -> Tuple[str, str, str]:
    try:
        text = clean_text(input_text)
        text = remove_run_repetition(text, max_run=3)

        if len(text.split()) < 80:
            return "Please upload/paste a longer document.", "", ""

        input_quality = text_quality_report(text)

        if input_quality.startswith("bad"):
            # Do not pass obviously bad input directly into LongT5.
            fallback, fbdebug = extract_legal_skeleton(text, max_sentences=10, use_minilm=False)
            return (
                fallback or "Input appears too noisy to summarize safely.",
                "",
                f"Input quality guard triggered: {input_quality}\n{fbdebug}"
            )

        if mode == "Fast":
            beams = 1
            section_summary_tokens = min(int(section_summary_tokens), 180)
            final_summary_tokens = min(int(final_summary_tokens), 360)
            anchors_per_block = min(int(anchors_per_block), 3)
        elif mode == "Balanced":
            beams = 2
        else:
            beams = 3

        # Detect sections or fallback to ordered windows.
        if force_ordered_windows:
            windows = split_token_windows(text, max_tokens=max_window_tokens, overlap_tokens=overlap_tokens)
            sections = [(f"Ordered Window {i+1}", w) for i, w in enumerate(windows)]
            section_debug = f"Forced ordered-window mode. Windows: {len(sections)}"
        else:
            sections, section_debug = detect_sections(text)

            if len(sections) == 1 and sections[0][0] == "Full Document":
                windows = split_token_windows(text, max_tokens=max_window_tokens, overlap_tokens=overlap_tokens)
                sections = [(f"Ordered Window {i+1}", w) for i, w in enumerate(windows)]
                section_debug += f"\nFallback converted Full Document into {len(sections)} ordered windows."

        if not sections:
            return "No valid sections/windows created.", "", ""

        section_summaries = []
        debug_parts = []

        for idx, (title, content) in enumerate(sections, start=1):
            summary, sdebug = summarize_section(
                title=title,
                content=content,
                anchor_method=anchor_method,
                anchors_per_block=anchors_per_block,
                max_window_tokens=max_window_tokens,
                overlap_tokens=overlap_tokens,
                section_summary_tokens=section_summary_tokens,
                reduce_input_tokens=reduce_input_tokens,
                beams=beams,
                minilm_weight=minilm_weight,
                tfidf_weight=tfidf_weight,
                use_skeleton=use_skeleton,
            )

            section_summaries.append((title, summary))
            debug_parts.append(f"[SECTION/WINDOW {idx}] {sdebug}")

        merged_section_summaries = "\n\n".join(
            f"[SECTION {idx}: {title}]\n{summary}"
            for idx, (title, summary) in enumerate(section_summaries, start=1)
        )

        # Final reduce if too large.
        current_blocks = [
            f"[SECTION: {title}]\n{summary}"
            for title, summary in section_summaries
            if summary.strip()
        ]

        final_reduce_rounds = 0
        while True:
            merged = "\n\n".join(current_blocks)

            if token_len(merged) <= int(reduce_input_tokens) or len(current_blocks) <= 1:
                break

            groups = group_token_safe(current_blocks, max_input_tokens=reduce_input_tokens)
            reduced = []

            for group in groups:
                prompt = (
                    "Merge these ordered legal/document section summaries into a faithful intermediate summary. "
                    "Do not invent facts. Preserve sequence and remove repetition.\n\n"
                    + group
                )

                red, rdebug = longt5_generate_guarded(
                    prompt=prompt,
                    fallback_text=group,
                    max_input_tokens=int(reduce_input_tokens),
                    max_output_tokens=max(220, int(section_summary_tokens)),
                    num_beams=int(beams),
                    min_new_tokens=80,
                    enable_retry=True,
                )

                reduced.append(red)
                debug_parts.append(f"final intermediate reduce: {rdebug}")

            current_blocks = [
                f"[INTERMEDIATE SUMMARY {i+1}]\n{s}"
                for i, s in enumerate(reduced)
            ]

            final_reduce_rounds += 1
            if final_reduce_rounds >= 4:
                break

        final_input = "\n\n".join(current_blocks)

        doc_skeleton = ""
        skeleton_debug = "Document skeleton disabled."
        if use_skeleton:
            doc_skeleton, skeleton_debug = extract_legal_skeleton(text, max_sentences=10, use_minilm=False)

        final_prompt = (
            "Create the final faithful summary from these ordered legal/document section summaries. "
            "Use clear structure when possible: Facts/Background, Issues, Arguments, Reasoning, Decision/Outcome. "
            "Do not add unsupported facts. Avoid repetition.\n\n"
        )

        if doc_skeleton:
            final_prompt += f"DOCUMENT FACTUAL SKELETON:\n{doc_skeleton}\n\n"

        final_prompt += f"ORDERED SECTION SUMMARIES:\n{final_input}"

        final_summary, final_gen_debug = longt5_generate_guarded(
            prompt=final_prompt,
            fallback_text=final_input if final_input.strip() else text,
            max_input_tokens=int(reduce_input_tokens),
            max_output_tokens=int(final_summary_tokens),
            num_beams=int(beams),
            min_new_tokens=max(100, int(final_summary_tokens) // 4),
            enable_retry=True,
        )

        # If even final summary is bad, return extractive legal skeleton.
        bad_final, bad_reason = is_bad_summary(final_summary)
        final_quality_msg = f"final_quality: {'bad' if bad_final else 'ok'} ({bad_reason})"

        if bad_final:
            fallback, fbdebug = extract_legal_skeleton(text, max_sentences=12, use_minilm=False)
            final_summary = fallback or final_summary
            final_quality_msg += f"\nfinal extractive fallback applied: {fbdebug}"

        debug = "\n".join([
            f"Device: {DEVICE}",
            f"LongT5 model: {SUM_MODEL_NAME}",
            f"Embedding model: {EMBED_MODEL_NAME}",
            "Architecture: Guarded Section-wise LongT5 + Extractive Skeleton",
            f"Mode: {mode}",
            f"Input words: {len(text.split())}",
            f"Input quality: {input_quality}",
            f"Sections/windows summarized: {len(sections)}",
            f"Max window tokens: {max_window_tokens}",
            f"Overlap tokens: {overlap_tokens}",
            f"Section summary tokens: {section_summary_tokens}",
            f"Reduce/final input tokens: {reduce_input_tokens}",
            f"Final summary tokens: {final_summary_tokens}",
            f"Beams: {beams}",
            f"Skeleton enabled: {use_skeleton}",
            f"Document skeleton: {skeleton_debug}",
            f"Anchor method: {anchor_method}",
            f"Anchors per block: {anchors_per_block if anchor_method != 'None' else 0}",
            f"MiniLM weight: {minilm_weight}",
            f"TF-IDF weight: {tfidf_weight}",
            f"Final reduce rounds: {final_reduce_rounds}",
            f"Final generation: {final_gen_debug}",
            final_quality_msg,
            "",
            "Section detection:",
            section_debug,
            "",
            "Detailed logs:",
            *debug_parts,
        ])

        return final_summary, merged_section_summaries, debug

    except Exception as e:
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        return (
            f"Summarization failed safely: {type(e).__name__}: {e}",
            "",
            "Try Fast mode, force ordered-window fallback, disable anchors, reduce token limits, or use T4/A10G GPU."
        )


Using device: cuda
GPU: Tesla T4


## Option A — Upload a PDF and extract text

Run this cell, upload a PDF, and it will extract cleaned text.


In [ ]:
# Cell 5A — Upload PDF and extract text
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded.keys()))

raw_text, extraction_debug = extract_text_from_pdf(pdf_path, max_pages=10)

print("PDF path:", pdf_path)
print("\n--- Extraction Debug ---")
print(extraction_debug)

print("\n--- First 2000 characters of extracted text ---")
print(raw_text[:2000])


Saving 138882021_2026-04-28.pdf to 138882021_2026-04-28.pdf
PDF path: 138882021_2026-04-28.pdf

--- Extraction Debug ---
Pages read: 10
Extracted words: 2023
Repeated header/footer lines removed: 2
Input quality: ok; words=2006, unique_ratio=0.32, top_word='the'(0.12), legal_hits=132, flags=none

--- First 2000 characters of extracted text ---
[ ] 2026 INSC 421 Reportable IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION Criminal Appeal No.558 of 2021 Sadek Ali @ Md. Sadek Ali and Anr. ...Appellants Versus The State of Assam and Anr. ...Respondents With Criminal Appeal No.850 of 2021 Criminal Appeal No.1264 of 2021 Criminal Appeal No.1428 of 2021 Criminal Appeal No.1096 of 2021 Criminal Appeal No.852 of 2022 Criminal Appeal No.266 of 2023 Criminal Appeal No…..…… of 2026 (@SLP (Crl.) No………….of 2026 @ Diary No.46790 of 2024) J U D G M E N T K. Vinod Chandran, J. An inept investigation or a scripted enquiry, both are fatal to criminal prosecution; but the latter has lethal con

## Option B — Paste text manually

Use this only if you do not want to upload a PDF.  
Replace the sample text with your own document text.


In [ ]:
# Cell 5B — Optional manual text input
# Skip this if you used PDF upload above.

# raw_text = '''
# Paste your long document text here.
# '''
# print("Manual text words:", len(raw_text.split()))


## Run summarization directly

Start with these fast settings. After it works, increase `max_pages`, `final_summary_tokens`, or switch `mode` from `Fast` to `Balanced`.


In [ ]:
# Cell 6 — Run summarization without Gradio

final_summary, section_summaries, debug_log = guarded_sectionwise_summarize(
    input_text=raw_text,
    mode="Fast",                    # Fast / Balanced / Quality
    force_ordered_windows=True,      # Safer for noisy PDFs
    anchor_method="None",            # None / K-Means / MMR
    anchors_per_block=3,
    use_skeleton=True,
    max_window_tokens=2048,
    overlap_tokens=64,
    section_summary_tokens=160,
    reduce_input_tokens=2048,
    final_summary_tokens=320,
    minilm_weight=0.60,
    tfidf_weight=0.40,
)

print("\n================ FINAL SUMMARY ================\n")
print(final_summary)

print("\n================ SECTION / WINDOW SUMMARIES ================\n")
print(section_summaries[:5000])

print("\n================ DEBUG LOG START ================\n")
print(debug_log[:5000])


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/295 [00:00<?, ?it/s]

[transformers] LongT5ForConditionalGeneration LOAD REPORT from: pszemraj/long-t5-tglobal-base-16384-book-summary
Key                         | Status  | 
----------------------------+---------+-
encoder.embed_tokens.weight | MISSING | 
decoder.embed_tokens.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=8) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documen


================ FINAL SUMMARY ================

1. Sadek Ali and Anr. ...Appellants Versus The State of Assam and Anr. ...Respondents With Criminal Appeal No.850 of 2021 Criminal Appeal No.1264 of 2021 Criminal Appeal No.1428 of 2021 Criminal Appeal No.1096 of 2021 Criminal Appeal No.852 of 2022 Criminal Appeal No.266 of 2023 Criminal Appeal No........... of 2026 (@SLP (Crl.) No.............of 2026 @ Diary No.46790 of 2024) J U D G M E N T K.
2. Eighteen witnesses were examined before the trial court, of which six were eyewitnesses: one disbelieved by the trial court and the High Court.
3. The High Court extracted and approved the findings of the trial court that the witnesses were examined on the strength of the GD entry, especially PW2 and PW13 who were eyewitnesses.
4. The High Court found that the attempt of the appellants’ counsel to decry the investigation on the ground of a delayed FIR falls flat, since there was already a GD entry recorded in the jurisdictional [ ] Police Sta

## Save results as text files

This creates downloadable `.txt` files for summary, section summaries, and debug logs.


In [ ]:
# Cell 7 — Save outputs

with open("final_summary.txt", "w", encoding="utf-8") as f:
    f.write(final_summary)

with open("section_summaries.txt", "w", encoding="utf-8") as f:
    f.write(section_summaries)

with open("debug_log.txt", "w", encoding="utf-8") as f:
    f.write(debug_log)

print("Saved:")
print("- final_summary.txt")
print("- section_summaries.txt")
print("- debug_log.txt")


Saved:
- final_summary.txt
- section_summaries.txt
- debug_log.txt


In [ ]:
# Cell 8 — Download output files from Colab
from google.colab import files

files.download("final_summary.txt")
files.download("section_summaries.txt")
files.download("debug_log.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Stronger test settings

After the first run works, try:

```python
mode="Balanced"
force_ordered_windows=False
max_window_tokens=3072
section_summary_tokens=260
reduce_input_tokens=3072
final_summary_tokens=576
```

Use `Quality` mode only after confirming memory is stable.


In [ ]:
# Install evaluation libraries
!pip install -q nltk rouge_score bert_score

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab') # Ensure punkt_tab is downloaded

from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
from bert_score import score

# --- Define Candidate Summary ---
candidate_summary = final_summary

# --- Define Reference Text ---
# For a more meaningful evaluation, especially for ROUGE and BLEU,
# it is recommended to compare your generated summary against a concise,
# human-written reference summary, rather than the entire raw document.
# You can replace `raw_text` with your actual reference summary if available.

# Example of using a custom reference summary:
# reference_summary_text = "Your human-written reference summary here."
# reference_for_metrics = reference_summary_text

# For this demonstration, we'll use raw_text as a very long reference.
# This might not yield ideal scores but demonstrates the functionality.
reference_for_metrics = raw_text

print("\n--- Calculating Evaluation Metrics ---\n")

# --- BLEU Score ---
# BLEU (Bilingual Evaluation Understudy) measures the n-gram overlap between a candidate and reference.
# It requires tokenized sentences.
candidate_tokens = nltk.word_tokenize(candidate_summary.lower())
reference_tokens = [nltk.word_tokenize(reference_for_metrics.lower())]

# BLEU-1, BLEU-2, BLEU-3, BLEU-4 are common (unigram, bigram, trigram, quadgram precision)
bleu1 = sentence_bleu(reference_tokens, candidate_tokens, weights=(1, 0, 0, 0))
bleu2 = sentence_bleu(reference_tokens, candidate_tokens, weights=(0.5, 0.5, 0, 0))
bleu3 = sentence_bleu(reference_tokens, candidate_tokens, weights=(0.33, 0.33, 0.33, 0))
bleu4 = sentence_bleu(reference_tokens, candidate_tokens, weights=(0.25, 0.25, 0.25, 0.25))

print(f"BLEU-1: {bleu1:.4f}")
print(f"BLEU-2: {bleu2:.4f}")
print(f"BLEU-3: {bleu3:.4f}")
print(f"BLEU-4: {bleu4:.4f}")

# --- ROUGE Score ---
# ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures overlap
# using n-grams, word sequences, and word pairs.
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = scorer.score(reference_for_metrics, candidate_summary)

print("\nROUGE Scores:")
for key, value in rouge_scores.items():
    print(f"{key}: P={value.precision:.4f}, R={value.recall:.4f}, F={value.fmeasure:.4f}")

# --- BERTScore ---
# BERTScore leverages pre-trained BERT embeddings to compute a similarity score.
# It's generally more robust to paraphrasing than n-gram based metrics.
# This might take a moment to download the BERT model if not cached.

# Ensure candidate_summary and reference_for_metrics are not empty
if not candidate_summary.strip() or not reference_for_metrics.strip():
    print("\nBERTScore requires non-empty candidate and reference texts. Skipping BERTScore.")
else:
    # BERTScore expects lists of strings
    P, R, F1 = score([candidate_summary], [reference_for_metrics], lang='en', verbose=True)
    print("\nBERTScore:")
    print(f"Precision: {P.mean():.4f}")
    print(f"Recall: {R.mean():.4f}")
    print(f"F1 Score: {F1.mean():.4f}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



--- Calculating Evaluation Metrics ---

BLEU-1: 0.0099
BLEU-2: 0.0097
BLEU-3: 0.0096
BLEU-4: 0.0094

ROUGE Scores:
rouge1: P=0.9840, R=0.1782, F=0.3018
rouge2: P=0.9598, R=0.1734, F=0.2938
rougeL: P=0.9786, R=0.1772, F=0.3001


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.47 seconds, 2.14 sentences/sec

BERTScore:
Precision: 0.8707
Recall: 0.8597
F1 Score: 0.8652
